In [26]:
import boto3
from datetime import date
import random
import pandas as pd
import os
from PIL import Image
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torchvision.transforms import ToPILImage

In [27]:
today_str = '2025-09-24'

s3 = boto3.client('s3', region_name='ap-northeast-2')


bucket_name = 'real-estate-avm'

object_key = f'raw/satellite-imagery/dt={today_str}/'
local_file = 'data/file.csv'


In [28]:
prefix = f'raw/satellite-imagery/dt={today_str}/'   # 예: "images/"

local_dir = 'data/'

response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

for obj in response.get('Contents', []):
    key = obj['Key']
    filename = key.split('/')[-1]
    s3.download_file(bucket_name, key, os.path.join(local_dir, filename))

In [ ]:
output_folder = "data/"

base_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.0, 0.0, 0.0],
        std=[1.0, 1.0, 1.0]
    )
])


In [30]:
to_pil = ToPILImage()
filenames = [f for f in os.listdir(output_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

In [32]:
for filename in filenames:
    img_path = os.path.join(output_folder,filename)
    img = Image.open(img_path).convert("RGB")

    degree = random.uniform(-90, 90)
    rotated_img = TF.rotate(img, degree, fill=(255, 255, 255))

    augmented_img = base_transform(rotated_img)
    augmented_img_pil = to_pil(augmented_img)

    save_path = os.path.join('data', filename)
    augmented_img_pil.save(save_path)